In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from typing import Union, Optional, Dict
import pyranges as pr

In [2]:
def check_mem(df):
    # 1. Show memory per column (including object-dtypes like strings)
    mem_per_col = df.memory_usage(index=True, deep=True)
    print(mem_per_col)
    
    # 2. Sum it up for the total footprint
    total = mem_per_col.sum()
    print(f"\nTotal memory usage: {total/1024**2:.2f} MB")


In [3]:
df = pd.read_csv("/ceph/MethDev/pbio/kay/data/annotated_full_col.CG_2.fast.tsv", sep="\t")
df

,cluster,chr,start,end,score,c,t,n,flag_euc_gene,flag_het_gene,flag_euc_TE,flag_het_TE
0,0,1,101,200,0.8951,350,41,6,False,False,False,False
1,0,1,301,400,0.5487,62,51,2,False,False,False,False
2,0,1,401,500,0.8246,47,10,1,False,False,False,False
3,0,1,501,600,0.7206,98,38,3,False,False,False,False
4,0,1,601,700,0.8982,203,23,6,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
16258583,9,5,26974801,26974900,0.8000,32,8,10,False,False,False,False
16258584,9,5,26974901,26975000,1.0000,4,0,2,False,False,False,False
16258585,9,5,26975101,26975200,1.0000,4,0,2,False,False,False,False
16258586,9,5,26975201,26975300,0.9773,43,1,14,False,False,False,False


In [4]:
df['cluster'].unique()

array([ 0,  1, 10, 11, 12, 13, 14, 15, 16,  2,  3,  4,  5,  6,  7,  8,  9])

In [5]:
check_mem(df)

Index                  128
cluster          130068704
chr              130068704
start            130068704
end              130068704
score            130068704
c                130068704
t                130068704
n                130068704
flag_euc_gene     16258588
flag_het_gene     16258588
flag_euc_TE       16258588
flag_het_TE       16258588
dtype: int64

Total memory usage: 1054.37 MB


In [6]:
low_cov = (df["c"] + df["t"]) < 5
df["score_masked"] = df["score"]
df.loc[low_cov, "score_masked"] = np.nan

In [7]:
df

,cluster,chr,start,end,score,c,t,n,flag_euc_gene,flag_het_gene,flag_euc_TE,flag_het_TE,score_masked
0,0,1,101,200,0.8951,350,41,6,False,False,False,False,0.8951
1,0,1,301,400,0.5487,62,51,2,False,False,False,False,0.5487
2,0,1,401,500,0.8246,47,10,1,False,False,False,False,0.8246
3,0,1,501,600,0.7206,98,38,3,False,False,False,False,0.7206
4,0,1,601,700,0.8982,203,23,6,False,False,False,False,0.8982
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16258583,9,5,26974801,26974900,0.8000,32,8,10,False,False,False,False,0.8000
16258584,9,5,26974901,26975000,1.0000,4,0,2,False,False,False,False,NaN
16258585,9,5,26975101,26975200,1.0000,4,0,2,False,False,False,False,NaN
16258586,9,5,26975201,26975300,0.9773,43,1,14,False,False,False,False,0.9773


In [8]:
window_max = (
    df
    .groupby(["chr","start","end"], sort=False)["score_masked"]
    .transform("max")
)

In [9]:
window_max

0           1.0000
1           1.0000
2           1.0000
3           1.0000
4           1.0000
             ...  
16258583    0.9892
16258584    1.0000
16258585    1.0000
16258586    1.0000
16258587    1.0000
Name: score_masked, Length: 16258588, dtype: float64

In [10]:
keep_mask = window_max >= 0.2

In [11]:
#df[(df['chr']==5) & (df['start']==26974801)]

In [12]:
#check_mem(df)

In [13]:
# 4a) To get the rows you want to **keep**:
df_keep = df[keep_mask].reset_index(drop=True)

# 4b) To get the rows you want to **drop**:
df_drop = df[~keep_mask].reset_index(drop=True)


In [14]:
df_keep

,cluster,chr,start,end,score,c,t,n,flag_euc_gene,flag_het_gene,flag_euc_TE,flag_het_TE,score_masked
0,0,1,101,200,0.8951,350,41,6,False,False,False,False,0.8951
1,0,1,301,400,0.5487,62,51,2,False,False,False,False,0.5487
2,0,1,401,500,0.8246,47,10,1,False,False,False,False,0.8246
3,0,1,501,600,0.7206,98,38,3,False,False,False,False,0.7206
4,0,1,601,700,0.8982,203,23,6,False,False,False,False,0.8982
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5925984,9,5,26974801,26974900,0.8000,32,8,10,False,False,False,False,0.8000
5925985,9,5,26974901,26975000,1.0000,4,0,2,False,False,False,False,NaN
5925986,9,5,26975101,26975200,1.0000,4,0,2,False,False,False,False,NaN
5925987,9,5,26975201,26975300,0.9773,43,1,14,False,False,False,False,0.9773


In [15]:
check_mem(df_keep)

Index                 128
cluster          47407912
chr              47407912
start            47407912
end              47407912
score            47407912
c                47407912
t                47407912
n                47407912
flag_euc_gene     5925989
flag_het_gene     5925989
flag_euc_TE       5925989
flag_het_TE       5925989
score_masked     47407912
dtype: int64

Total memory usage: 429.51 MB


In [16]:
# this will reorder clusters 0→1→2→…→16, 
# but within each cluster the rows stay in the same order as before
df_keep = df_keep.sort_values(
    by="cluster",
    kind="mergesort"
).reset_index(drop=True)


In [17]:
df_keep

,cluster,chr,start,end,score,c,t,n,flag_euc_gene,flag_het_gene,flag_euc_TE,flag_het_TE,score_masked
0,0,1,101,200,0.8951,350,41,6,False,False,False,False,0.8951
1,0,1,301,400,0.5487,62,51,2,False,False,False,False,0.5487
2,0,1,401,500,0.8246,47,10,1,False,False,False,False,0.8246
3,0,1,501,600,0.7206,98,38,3,False,False,False,False,0.7206
4,0,1,601,700,0.8982,203,23,6,False,False,False,False,0.8982
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5925984,16,5,26974801,26974900,0.8462,22,4,7,False,False,False,False,0.8462
5925985,16,5,26974901,26975000,0.7500,3,1,2,False,False,False,False,NaN
5925986,16,5,26975101,26975200,1.0000,4,0,4,False,False,False,False,NaN
5925987,16,5,26975201,26975300,1.0000,8,0,8,False,False,False,False,1.0000


In [18]:
check_mem(df_keep)

Index                 128
cluster          47407912
chr              47407912
start            47407912
end              47407912
score            47407912
c                47407912
t                47407912
n                47407912
flag_euc_gene     5925989
flag_het_gene     5925989
flag_euc_TE       5925989
flag_het_TE       5925989
score_masked     47407912
dtype: int64

Total memory usage: 429.51 MB


In [19]:
out_path = f"/ceph/MethDev/pbio/kay/data/annotated_filtered_col.CG_2.fast.tsv"
df_keep.to_csv(out_path, sep="\t", index=False)
print(f"Wrote: {out_path}")


Wrote: ./data/annotated_filtered_col.CG_2.fast.tsv
